<a href="https://colab.research.google.com/github/Bunkhuoch-Ann/Side_Quests_simple/blob/main/3_Body_simulation.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
"""
Three-Body Gravitational Simulation
=====================================
Two bodies start as a bound binary; a third (lighter) body perturbs the
system from a distance. Integrated with RK4. Produces a 3D animated MP4
showing the bodies and their traced orbital trajectories, plus a 2D
"top-down" panel and an energy-conservation panel (the imshow-style
diagnostic panel).

Physics
-------
Newtonian gravity, G=1 (natural units), softened potential to avoid
singularities during close encounters:

    a_i = sum_j  G * m_j * (r_j - r_i) / (|r_j - r_i|^2 + eps^2)^(3/2)

Usage
-----
    python three_body_sim.py

Tweak the INITIAL CONDITIONS section below to explore different outcomes.
"""

import numpy as np
import matplotlib
matplotlib.use("Agg")  # headless rendering
import matplotlib.pyplot as plt
from matplotlib import animation
from mpl_toolkits.mplot3d import Axes3D  # noqa: F401 (registers 3D projection)

# ──────────────────────────────────────────────────────────────────────────

In [ ]:
# ──────────────────────────────────────────────────────────────────────────
# CONSTANTS
# ──────────────────────────────────────────────────────────────────────────
G = 1.0
SOFTENING = 0.0005
EJECTION_DIST = 100

COLORS = ["#3b82f6", "#ec4899", "#10b981"]  # blue, pink, green
LABELS = ["Body A", "Body B", "Perturber C"]

In [ ]:
# ──────────────────────────────────────────────────────────────────────────
# PHYSICS: RK4 integrator
# ──────────────────────────────────────────────────────────────────────────
def accelerations(pos, masses):
    """pos: (3,3) array of [x,y,z] per body. Returns (3,3) accelerations."""
    acc = np.zeros_like(pos)
    for i in range(3):
        for j in range(3):
            if i == j:
                continue
            r_vec = pos[j] - pos[i]
            r2 = r_vec @ r_vec + SOFTENING ** 2
            r3 = r2 ** 1.5
            acc[i] += G * masses[j] * r_vec / r3
    return acc


def rk4_step(pos, vel, masses, dt):
    def deriv(p, v):
        return v, accelerations(p, masses)

    k1p, k1v = deriv(pos, vel)
    k2p, k2v = deriv(pos + 0.5 * dt * k1p, vel + 0.5 * dt * k1v)
    k3p, k3v = deriv(pos + 0.5 * dt * k2p, vel + 0.5 * dt * k2v)
    k4p, k4v = deriv(pos + dt * k3p, vel + dt * k3v)

    new_pos = pos + (dt / 6.0) * (k1p + 2 * k2p + 2 * k3p + k4p)
    new_vel = vel + (dt / 6.0) * (k1v + 2 * k2v + 2 * k3v + k4v)
    return new_pos, new_vel


def total_energy(pos, vel, masses):
    ke = 0.5 * np.sum(masses[:, None] * vel ** 2)
    pe = 0.0
    for i in range(3):
        for j in range(i + 1, 3):
            r = np.linalg.norm(pos[j] - pos[i])
            pe -= G * masses[i] * masses[j] / np.sqrt(r ** 2 + SOFTENING ** 2)
    return ke + pe



In [ ]:
# ──────────────────────────────────────────────────────────────────────────
# INITIAL CONDITIONS  (edit these to explore different scenarios)
# ──────────────────────────────────────────────────────────────────────────
def build_initial_conditions(
    mA=1.0, mB=1.0, mC=0.3,
    sep_binary=3.0, sep_perturber=14.0,
    pert_v=(0.0, -0.55, 0.10),
):
    """Two bodies in a circular binary about their common COM (in the xy
    plane), plus a perturber approaching from +y with given velocity."""
    masses = np.array([mA, mB, mC])

    rA = (mB / (mA + mB)) * sep_binary
    rB = (mA / (mA + mB)) * sep_binary
    v_orbit = np.sqrt(G * (mA + mB) / sep_binary)
    vA = (mB / (mA + mB)) * v_orbit
    vB = (mA / (mA + mB)) * v_orbit

    pos = np.array([
        [-rA, 0.0, 0.0],
        [rB, 0.0, 0.0],
        # [0.0, sep_perturber, 0.0],
        [0.0, 0.0, sep_perturber],
    ])
    vel = np.array([
        [0.0, -vA, 0.0],
        [0.0, vB, 0.0],
        list(pert_v),
    ])
    return pos, vel, masses



In [ ]:
# ──────────────────────────────────────────────────────────────────────────
# RUN SIMULATION
# ──────────────────────────────────────────────────────────────────────────
from tqdm.notebook import tqdm # Import tqdm for internal progress bar
import numpy as np # Ensure numpy is imported

def run_simulation(pos, vel, masses, dt=0.0015, n_steps=6000, record_every=2,
                   energy_threshold_percent=20.0, dt_refinement_factor=0.5, max_dt_substeps=5):
    """
    Integrate forward and record trajectory history + energy with adaptive time-stepping.
    When energy conservation is violated by more than energy_threshold_percent,
    the time step (dt) is refined by dt_refinement_factor until conservation is met
    or max_dt_substeps is reached for that single integration step.
    """
    traj_list = []
    energy_list = []
    times_list = []

    ejected_body = None
    ejected_at = None

    t = 0.0 # Current simulation time
    original_dt = dt # The desired time step size for each conceptual integration step

    current_pos = np.copy(pos)
    current_vel = np.copy(vel)

    # Initial record
    traj_list.append(np.copy(current_pos))
    energy_list.append(total_energy(current_pos, current_vel, masses))
    times_list.append(t)

    pbar = tqdm(total=n_steps, desc="RK4 Integration Steps")

    # Iterate for a fixed number of 'conceptual' steps, each aiming to advance by original_dt
    for step_idx in range(n_steps):
        # Keep track of the energy from the last *successfully completed* full 'original_dt' conceptual step.
        prev_recorded_energy = energy_list[-1]

        # Use temporary variables for the state within this conceptual step (to allow rollback)
        temp_pos = np.copy(current_pos)
        temp_vel = np.copy(current_vel)
        temp_t = t # Current time before this conceptual step starts

        time_remaining_for_this_conceptual_step = original_dt
        substep_refinement_attempts = 0

        while time_remaining_for_this_conceptual_step > 1e-12 * original_dt: # While we still need to cover time for this conceptual step
            # Determine the dt for the current substep attempt
            dt_substep_to_try = original_dt * (dt_refinement_factor ** substep_refinement_attempts)
            # Ensure we don't overstep the remaining time for this conceptual step
            dt_substep_to_try = min(dt_substep_to_try, time_remaining_for_this_conceptual_step)

            # Avoid extremely small dt that could lead to infinite loops or precision issues
            if dt_substep_to_try < 1e-18 * original_dt or dt_substep_to_try < 1e-18: # Absolute minimum dt
                pbar.write(f"Warning: dt_substep_to_try became extremely small ({dt_substep_to_try:.2e}) at conceptual step {step_idx}. Skipping remaining time for this step.")
                break # Break out of substep loop if dt becomes too small

            # Perform the RK4 substep from the current temporary state
            trial_pos, trial_vel = rk4_step(temp_pos, temp_vel, masses, dt_substep_to_try)
            trial_energy = total_energy(trial_pos, trial_vel, masses)

            # Only calculate energy change if prev_recorded_energy is not zero to avoid division by zero
            if abs(prev_recorded_energy) > 1e-12: # A small epsilon to check for near-zero energy
                energy_change_percent = abs(trial_energy - prev_recorded_energy) / abs(prev_recorded_energy) * 100
            else:
                # If initial energy is zero, any non-zero trial_energy means a significant change
                energy_change_percent = 0.0 # Or a very large number if any change is unacceptable
                if abs(trial_energy) > 1e-12: energy_change_percent = float('inf')

            if energy_change_percent <= energy_threshold_percent:
                # Substep was successful: update temporary state and remaining time
                temp_pos = trial_pos
                temp_vel = trial_vel
                temp_t += dt_substep_to_try
                time_remaining_for_this_conceptual_step -= dt_substep_to_try
                substep_refinement_attempts = 0 # Reset refinements for the next chunk of this conceptual step
                pbar.set_postfix_str("") # Clear refinement message
            else:
                # Substep caused too large an energy change: refine dt and retry
                substep_refinement_attempts += 1
                if substep_refinement_attempts >= max_dt_substeps:
                    pbar.write(f"Warning: Energy conservation threshold not met for conceptual step {step_idx} even after {max_dt_substeps} refinements. Energy change: {energy_change_percent:.2f}% (last dt_substep_to_try: {dt_substep_to_try:.2e}). Accepting step.")
                    # If max refinements reached, accept the current trial state and move on
                    temp_pos = trial_pos # Apply the last (best effort) trial
                    temp_vel = trial_vel
                    temp_t += dt_substep_to_try
                    time_remaining_for_this_conceptual_step -= dt_substep_to_try
                    substep_refinement_attempts = 0 # Reset for next conceptual step
                    pbar.set_postfix_str("") # Clear refinement message
                    break # Break out of substep loop, proceed to recording for conceptual step
                else:
                    pbar.set_postfix_str(f"Refining dt (attempt {substep_refinement_attempts}, current dt_substep:{dt_substep_to_try:.2e})")
                    # temp_pos, temp_vel, temp_t are NOT updated, effectively retrying this segment

        # After the conceptual step is complete (or max refinements hit), update global state
        current_pos = temp_pos
        current_vel = temp_vel
        t = temp_t # Global simulation time advances

        # Record data if required
        if step_idx % record_every == 0:
            traj_list.append(np.copy(current_pos))
            energy_list.append(total_energy(current_pos, current_vel, masses)) # Recalculate energy for recorded state
            times_list.append(t)

        # Ejection check
        if ejected_body is None:
            com_total = (masses[:, None] * current_pos).sum(axis=0) / masses.sum()
            for i in range(3):
                others = [j for j in range(3) if j != i]
                com_other = (masses[others[0]] * current_pos[others[0]] +
                             masses[others[1]] * current_pos[others[1]]) / (masses[others[0]] + masses[others[1]])
                r_i = np.linalg.norm(current_pos[i] - com_other)
                r_other = np.linalg.norm(current_pos[others[0]] - current_pos[others[1]])
                if r_i > EJECTION_DIST and r_other < 10.0:
                    ejected_body = i
                    ejected_at = t
                    break

        pbar.update(1)

    pbar.close()

    # Convert lists to numpy arrays
    traj_arr = np.array(traj_list)
    energy_arr = np.array(energy_list)
    times_arr = np.array(times_list)

    # Return the final pos and vel for continuation, along with the history
    return traj_arr, energy_arr, times_arr, ejected_body, ejected_at, current_pos, current_vel

def run_simulation1(pos, vel, masses, dt=0.0015, n_steps=6000, record_every=2):
    """
    Integrate forward and record trajectory history + energy using a fixed time step (dt).
    This version does NOT include adaptive time-stepping or energy conservation checks
    for faster execution, suitable for demonstrations where accuracy over long periods
    is less critical than speed.
    """
    traj_list = []
    energy_list = []
    times_list = []

    ejected_body = None
    ejected_at = None

    t = 0.0 # Current simulation time
    current_pos = np.copy(pos)
    current_vel = np.copy(vel)

    # Initial record
    traj_list.append(np.copy(current_pos))
    energy_list.append(total_energy(current_pos, current_vel, masses))
    times_list.append(t)

    pbar = tqdm(total=n_steps, desc="RK4 Integration Steps (Fixed DT)")

    for step_idx in range(n_steps):
        # Perform the RK4 step with the fixed time step dt
        current_pos, current_vel = rk4_step(current_pos, current_vel, masses, dt)
        t += dt # Advance global time by the fixed dt

        # Record data if required
        if step_idx % record_every == 0:
            traj_list.append(np.copy(current_pos))
            energy_list.append(total_energy(current_pos, current_vel, masses))
            times_list.append(t)

        # Ejection check
        if ejected_body is None:
            # This assumes that EJECTION_DIST and other related constants are globally available
            # or passed in. Based on the original function, they are global.
            com_total = (masses[:, None] * current_pos).sum(axis=0) / masses.sum()
            for i in range(3):
                others = [j for j in range(3) if j != i]
                com_other = (masses[others[0]] * current_pos[others[0]] +
                             masses[others[1]] * current_pos[others[1]]) / (masses[others[0]] + masses[others[1]])
                r_i = np.linalg.norm(current_pos[i] - com_other)
                r_other = np.linalg.norm(current_pos[others[0]] - current_pos[others[1]])
                if r_i > EJECTION_DIST and r_other < 10.0:
                    ejected_body = i
                    ejected_at = t
                    break

        pbar.update(1)

    pbar.close()

    # Convert lists to numpy arrays
    traj_arr = np.array(traj_list)
    energy_arr = np.array(energy_list)
    times_arr = np.array(times_list)

    # Return the final pos and vel for continuation, along with the history
    return traj_arr, energy_arr, times_arr, ejected_body, ejected_at, current_pos, current_vel

In [ ]:
import numpy as np
import matplotlib
matplotlib.use("Agg")  # headless rendering
import matplotlib.pyplot as plt
from matplotlib import animation
from mpl_toolkits.mplot3d import Axes3D  # noqa: F401 (registers 3D projection)
from tqdm.notebook import tqdm # Import tqdm

# ──────────────────────────────────────────────────────────────────────────
# VIDEO RENDERING
# ──────────────────────────────────────────────────────────────────────────
def render_video(traj, energy, times, masses, ejected_body, output_path,
                  trail_len=350, fps=30, n_frames_out=600, progress_callback=None,
                  initial_conditions_str=""): # Added new parameter
    """
    Renders a 3-panel video:
      - Left (large): 3D view with traced trajectories
      - Top right: top-down (x-y) projection, like an imshow/bird's-eye map
      - Bottom right: energy-conservation diagnostic over time
    """
    n_total = traj.shape[0]
    # Subsample to n_frames_out for a manageable video length
    idx = np.linspace(0, n_total - 1, min(n_frames_out, n_total)).astype(int)
    traj_s = traj[idx]
    energy_s = energy[idx]
    times_s = times[idx]

    fig = plt.figure(figsize=(13, 7), facecolor="#050a12")
    gs = fig.add_gridspec(2, 3, width_ratios=[2.2, 1, 1], height_ratios=[1, 1],
                           hspace=0.35, wspace=0.3)

    ax3d = fig.add_subplot(gs[:, 0], projection="3d", facecolor="#050a12")
    ax_top = fig.add_subplot(gs[0, 1:], facecolor="#0a111c")
    ax_energy = fig.add_subplot(gs[1, 1:], facecolor="#0a111c")

    # Style helper
    def style_3d(ax):
        ax.set_facecolor("#050a12")
        ax.xaxis.set_pane_color((0.02, 0.04, 0.07, 1.0))
        ax.yaxis.set_pane_color((0.02, 0.04, 0.07, 1.0))
        ax.zaxis.set_pane_color((0.02, 0.04, 0.07, 1.0))
        ax.tick_params(colors="#475569", labelsize=7)
        ax.xaxis.label.set_color("#64748b")
        ax.yaxis.label.set_color("#64748b")
        ax.zaxis.label.set_color("#64748b")
        for axis in (ax.xaxis, ax.yaxis, ax.zaxis):
            axis._axinfo["grid"]["color"] = (1, 1, 1, 0.06)

    style_3d(ax3d)
    ax3d.set_xlabel("X")
    ax3d.set_ylabel("Y")
    ax3d.set_zlabel("Z")

    def get_adaptive_limits(frame_i, margin=2.0, min_span=6.0, max_span=None, smooth_window=20):
        lo = max(0, frame_i - smooth_window)
        hi = min(len(traj_s), frame_i + smooth_window + 1)

        pts = traj_s[lo:hi].reshape(-1, 3)

        center = pts.mean(axis=0)

        span_x = pts[:, 0].max() - pts[:, 0].min()
        span_y = pts[:, 1].max() - pts[:, 1].min()
        span_z = pts[:, 2].max() - pts[:, 2].min()

        span = max(span_x, span_y, span_z) + margin
        span = max(span, min_span)

        if max_span is not None:
            span = min(span, max_span)

        half = span / 2.0

        return (
            (center[0] - half, center[0] + half),
            (center[1] - half, center[1] + half),
            (center[2] - half, center[2] + half),
        )

    all_pts = traj_s.reshape(-1, 3)
    pad = 2.0
    xlim = (all_pts[:, 0].min() - pad, all_pts[:, 0].max() + pad)
    ylim = (all_pts[:, 1].min() - pad, all_pts[:, 1].max() + pad)
    zlim = (all_pts[:, 2].min() - pad, all_pts[:, 2].max() + pad)

    # Top-down panel styling
    ax_top.set_title("Top-down (X-Y) trace", color="#94a3b8", fontsize=9, loc="left")
    ax_top.set_xlim(xlim)
    ax_top.set_ylim(ylim)
    ax_top.tick_params(colors="#475569", labelsize=7)
    for spine in ax_top.spines.values():
        spine.set_color("#1e293b")
    ax_top.set_aspect("equal")

    # Energy panel styling
    ax_energy.set_title("Total energy (conservation check)", color="#94a3b8", fontsize=9, loc="left")
    ax_energy.set_xlim(times_s[0], times_s[-1])
    e_pad = 0.05 * (energy_s.max() - energy_s.min() + 1e-9)
    ax_energy.set_ylim(energy_s.min() - e_pad, energy_s.max() + e_pad)
    ax_energy.tick_params(colors="#475569", labelsize=7)
    for spine in ax_energy.spines.values():
        spine.set_color("#1e293b")
    ax_energy.set_xlabel("t", color="#64748b", fontsize=8)

    # --- MARKER SIZE CALCULATION ---
    # Calculate marker sizes based on mass ratio, with a base size and scaling factor
    max_mass = np.max(masses)
    min_display_size_3d = 5
    size_scale_3d = 5 # Increased scaling for better visibility of mass differences
    marker_sizes_3d = [min_display_size_3d + size_scale_3d * (m / max_mass) for m in masses]

    min_display_size_top = 4
    size_scale_top = 8 # Increased scaling for better visibility of mass differences
    marker_sizes_top = [min_display_size_top + size_scale_top * (m / max_mass) for m in masses]
    # --------------------------------

    # Artists
    trail_lines_3d = [ax3d.plot([], [], [], color=COLORS[i], lw=1.2, alpha=0.8)[0] for i in range(3)]
    # Modified markersize to use calculated values
    points_3d = [ax3d.plot([], [], [], "o", color=COLORS[i], markersize=marker_sizes_3d[i],
                            markeredgecolor="white", markeredgewidth=0.4)[0] for i in range(3)]

    trail_lines_top = [ax_top.plot([], [], color=COLORS[i], lw=1.0, alpha=0.8)[0] for i in range(3)]
    # Modified markersize to use calculated values
    points_top = [ax_top.plot([], [], "o", color=COLORS[i], markersize=marker_sizes_top[i])[0] for i in range(3)]

    energy_line, = ax_energy.plot([], [], color="#60a5fa", lw=1.2)

    title_text = fig.text(0.02, 0.95, "", color="#e2e8f0", fontsize=13, fontweight="bold",
                           family="monospace")
    status_text = fig.text(0.02, 0.91, "", color="#f472b6", fontsize=10, family="monospace")
    # New text object for initial conditions
    ic_text = fig.text(0.02, 0.84, "", color="#64748b", fontsize=8, family="monospace", va="top")
    ic_text.set_text(initial_conditions_str) # Set the initial conditions text here

    # Update legend to reflect mass values and dynamic sizing
    legend_handles = [plt.Line2D([0], [0], marker="o", color="w", markerfacecolor=COLORS[i],
                                  markersize=marker_sizes_3d[i] * 0.8, # Scale legend marker size appropriately
                                  label=f"{LABELS[i]} (m={masses[i]:.2f})")
                       for i in range(3)]
    fig.legend(handles=legend_handles, loc="lower left", bbox_to_anchor=(0.02, 0.02),
               facecolor="#0a111c", edgecolor="#1e293b", labelcolor="#e2e8f0", fontsize=8)

    def init():
      cur_xlim, cur_ylim, cur_zlim = get_adaptive_limits(0, margin=3.0, min_span=5.0, max_span=40.0)

      ax3d.set_xlim(cur_xlim)
      ax3d.set_ylim(cur_ylim)
      ax3d.set_zlim(cur_zlim)

      ax_top.set_xlim(cur_xlim)
      ax_top.set_ylim(cur_ylim)

      return trail_lines_3d + points_3d + trail_lines_top + points_top + [energy_line, title_text, status_text, ic_text]

    def update(frame_i):
        ax3d.view_init(elev=22, azim=frame_i * 0.25)  # slow camera rotation
        # ax3d.view_init(elev=22, azim=45)

        cur_xlim, cur_ylim, cur_zlim = get_adaptive_limits(
            frame_i,
            margin=3.0,
            min_span=5.0,
            max_span=40.0,
            smooth_window=20
        )

        ax3d.set_xlim(cur_xlim)
        ax3d.set_ylim(cur_ylim)
        ax3d.set_zlim(cur_zlim)

        ax_top.set_xlim(cur_xlim)
        ax_top.set_ylim(cur_ylim)

        lo = max(0, frame_i - trail_len)
        for i in range(3):
            seg = traj_s[lo:frame_i + 1, i, :]
            trail_lines_3d[i].set_data(seg[:, 0], seg[:, 1])
            trail_lines_3d[i].set_3d_properties(seg[:, 2])
            points_3d[i].set_data([traj_s[frame_i, i, 0]], [traj_s[frame_i, i, 1]])
            points_3d[i].set_3d_properties([traj_s[frame_i, i, 2]])

            trail_lines_top[i].set_data(seg[:, 0], seg[:, 1])
            points_top[i].set_data([traj_s[frame_i, i, 0]], [traj_s[frame_i, i, 1]])

        energy_line.set_data(times_s[:frame_i + 1], energy_s[:frame_i + 1])

        title_text.set_text(f"Three-Body Simulation   t = {times_s[frame_i]:.2f}")
        if ejected_body is not None and times_s[frame_i] > 0:
            cur_r = np.linalg.norm(traj_s[frame_i, ejected_body])
            if cur_r > EJECTION_DIST * 0.6:
                status_text.set_text(f"★ {LABELS[ejected_body]} being ejected from the system")
            else:
                status_text.set_text("")
        else:
            status_text.set_text("")

        return trail_lines_3d + points_3d + trail_lines_top + points_top + [energy_line, title_text, status_text, ic_text]

    anim = animation.FuncAnimation(
        fig, update, frames=len(traj_s), init_func=init, blit=False, interval=1000 / fps
    )

    writer = animation.FFMpegWriter(fps=fps, bitrate=4000)
    anim.save(output_path, writer=writer, dpi=130, progress_callback=progress_callback)
    plt.close(fig)

In [ ]:
# ──────────────────────────────────────────────────────────────────────────
# MAIN
# ──────────────────────────────────────────────────────────────────────────
import os
from tqdm.notebook import tqdm # Import tqdm for progress bar
import numpy as np
import matplotlib
matplotlib.use("Agg")  # headless rendering
import matplotlib.pyplot as plt
from matplotlib import animation
from mpl_toolkits.mplot3d import Axes3D  # noqa: F401 (registers 3D projection)


if __name__ == "__main__":
    print("Building initial conditions...")
    # Capture initial condition parameters for display
    mA_val, mB_val, mC_val = 0.2, 0.5, 1
    sep_binary_val, sep_perturber_val = 4, 100
    pert_v_val = (0.01, 0.0, -0.3)

    pos0, vel0, masses = build_initial_conditions(
        mA=mA_val, mB=mB_val, mC=mC_val,
        sep_binary=sep_binary_val, sep_perturber=sep_perturber_val,
        pert_v=pert_v_val,
    )

    # Prepare the initial conditions string to display on the video
    initial_cond_str = (
        f"Initial Conditions:\n"
        f"  Masses: A={mA_val:.1f}, B={mB_val:.1f}, C={mC_val:.1f}\n"
        f"  Binary Sep: {sep_binary_val:.1f}, Perturber Sep: {sep_perturber_val:.1f}\n"
        f"  Perturber Vel: ({pert_v_val[0]:.2f}, {pert_v_val[1]:.2f}, {pert_v_val[2]:.2f})"
    )

    # --- User choice for simulation function ---
    use_adaptive_stepping = False # Set to False to use the faster run_simulation1
    # -------------------------------------------

    print("Running RK4 integration...")
    if use_adaptive_stepping:
        print("  Using run_simulation (with adaptive time-stepping).")
        traj, energy, times, ejected_body, ejected_at, final_pos, final_vel = run_simulation(
            pos0, vel0, masses, dt=1e-3, n_steps=300000, record_every=2,
            energy_threshold_percent=5e-8, dt_refinement_factor=0.01, max_dt_substeps=5
        )
    else:
        print("  Using run_simulation1 (fixed time-stepping).")
        traj, energy, times, ejected_body, ejected_at, final_pos, final_vel = run_simulation1(
            pos0, vel0, masses, dt=2e-3, n_steps=300000, record_every=2
        )

    # Make the final pos, vel, and masses available for continuation in subsequent cells
    pos = final_pos
    vel = final_vel
    # masses is already globally available if defined outside main, but good to be explicit if needed

    if ejected_body is not None:
        print(f"  -> {LABELS[ejected_body]} ejected at t = {ejected_at:.2f}")
    else:
        print("  -> No clean ejection detected in this run (system may still be chaotic/bound).")

    print(f"  -> Energy drift: {abs(energy[-1] - energy[0]) / abs(energy[0]) * 100:.4f}% "
          f"(should be small, e.g. < 1%)")

    print("Rendering video (this can take 1-2 min)....")
    output_filename = "three_body_simulation.mp4"

    # Calculate the actual number of frames that will be rendered by render_video
    # based on n_frames_out and the total available trajectory frames.
    n_frames_out_render = 300 # From the render_video call
    n_frames_to_render_total = min(n_frames_out_render, len(traj))

    with tqdm(total=n_frames_to_render_total, desc="Rendering video frames") as pbar_rendering:
        def progress_callback(frame_i, total_frames):
            pbar_rendering.update(1)

        render_video(traj, energy, times, masses, ejected_body,
                     output_path=output_filename,
                     trail_len=350, fps=30, n_frames_out=n_frames_out_render,
                     progress_callback=progress_callback,
                     initial_conditions_str=initial_cond_str) # Pass the initial conditions string

    print(f"Done. Saved to {output_filename}")

Building initial conditions...
Running RK4 integration...
  Using run_simulation1 (fixed time-stepping).


RK4 Integration Steps (Fixed DT):   0%|          | 0/300000 [00:00<?, ?it/s]

  -> Body B ejected at t = 532.24
  -> Energy drift: 0.0000% (should be small, e.g. < 1%)
Rendering video (this can take 1-2 min)....


Rendering video frames:   0%|          | 0/300 [00:00<?, ?it/s]

Done. Saved to three_body_simulation.mp4


Extending the simulation

In [ ]:
# ──────────────────────────────────────────────────────────────────────────
# CONTINUE SIMULATION AND RENDER EXTENDED VIDEO
# ──────────────────────────────────────────────────────────────────────────
import numpy as np
from tqdm.notebook import tqdm # Ensure tqdm is imported for progress bar

# These variables are available from the previous run of the MAIN cell (AIyZDwv7tzF8):
# traj, energy, times, ejected_body, ejected_at, pos (final pos from prev run), vel (final vel from prev run), masses

# Define parameters for the continuation
n_steps_extension = 2000  # Number of additional simulation steps
dt_extension = 1e-2         # Time step for the extension (should match previous dt if possible)
record_every_extension = 2  # Record frequency for the extension

# --- User choice for extended simulation function ---
use_adaptive_stepping_extension = True # Set to False to use the faster run_simulation1 for extension
# -----------------------------------------------------

print(f"Continuing simulation for an additional {n_steps_extension} steps...")

# Run the simulation again, starting from the final pos and vel of the previous run
if use_adaptive_stepping_extension:
    print("  Using run_simulation for extension (with adaptive time-stepping).")
    ext_traj, ext_energy, ext_times, ext_ejected_body, ext_ejected_at, final_pos_ext, final_vel_ext = run_simulation(
        pos, vel, masses, dt=dt_extension, n_steps=n_steps_extension, record_every=record_every_extension,
        energy_threshold_percent=5e-7, dt_refinement_factor=0.1, max_dt_substeps=5 # Using previous adaptive parameters
    )
else:
    print("  Using run_simulation1 for extension (fixed time-stepping).")
    ext_traj, ext_energy, ext_times, ext_ejected_body, ext_ejected_at, final_pos_ext, final_vel_ext = run_simulation1(
        pos, vel, masses, dt=dt_extension, n_steps=n_steps_extension, record_every=record_every_extension
    )

# Adjust the times of the extended simulation to be continuous with the previous run
# The 'times' array from the previous run contains time points up to its last recorded frame.
# 'ext_times' starts from t=0 for the new simulation segment.
# We need to shift 'ext_times' by the last time of the original simulation.
if len(times) > 0:
    time_offset = times[-1]
else:
    # If the original simulation had no recorded frames (unlikely but for robustness)
    time_offset = 0.0
ext_times_shifted = ext_times + time_offset

# Concatenate the original and extended simulation data
# Skip the first frame of ext_traj/ext_energy/ext_times if it exactly overlaps with the last of original
# This simple concatenation might have a small overlap if record_every was the same and the last frame aligned.
# For perfect continuity, we ensure no duplicate time points.

# Find where the extension should start without duplicating the last point if times are exactly aligned
start_idx_ext = 0
if len(times) > 0 and len(ext_times_shifted) > 0 and np.isclose(times[-1], ext_times_shifted[0]):
    start_idx_ext = 1 # Skip the first point of the extension if it's a duplicate of the last original point

combined_traj = np.concatenate((traj, ext_traj[start_idx_ext:]))
combined_energy = np.concatenate((energy, ext_energy[start_idx_ext:]))
combined_times = np.concatenate((times, ext_times_shifted[start_idx_ext:]))

# Update ejected body/time if ejection happened in the extended part
if ext_ejected_body is not None and ejected_body is None:
    ejected_body = ext_ejected_body
    ejected_at = ext_ejected_at + time_offset # Adjust ejection time
elif ejected_body is not None and ext_ejected_body is not None: # Ejection already happened, check if new one is earlier
    if ext_ejected_at + time_offset < ejected_at:
        ejected_body = ext_ejected_body
        ejected_at = ext_ejected_at + time_offset

# Recalculate energy drift for the combined run
if len(combined_energy) > 0:
    print(f"  -> Energy drift (combined): {abs(combined_energy[-1] - combined_energy[0]) / abs(combined_energy[0]) * 100:.4f}% "
          f"(should be small, e.g. < 1%)")
else:
    print("  -> No combined energy data to report drift.")

print("Rendering extended video (this can take 1-2 min)...")
output_filename_extended = "three_body_simulation_extended.mp4"

# Calculate the actual number of frames that will be rendered by render_video
n_frames_out_render_extended = 4800 # You can adjust this for the extended video
n_frames_to_render_total_extended = min(n_frames_out_render_extended, len(combined_traj))

with tqdm(total=n_frames_to_render_total_extended, desc="Rendering extended video frames") as pbar_rendering_ext:
    def progress_callback_ext(frame_i, total_frames):
        pbar_rendering_ext.update(1)

    render_video(combined_traj, combined_energy, combined_times, masses, ejected_body,
                 output_path=output_filename_extended,
                 trail_len=350, fps=240, n_frames_out=n_frames_out_render_extended,
                 progress_callback=progress_callback_ext,
                 initial_conditions_str=initial_cond_str)

print(f"Done. Extended simulation saved to {output_filename_extended}")

Continuing simulation for an additional 2000 steps...
  Using run_simulation for extension (with adaptive time-stepping).


RK4 Integration Steps:   0%|          | 0/2000 [00:00<?, ?it/s]

  -> Energy drift (combined): 0.0000% (should be small, e.g. < 1%)
Rendering extended video (this can take 1-2 min)...


Rendering extended video frames:   0%|          | 0/4800 [00:00<?, ?it/s]

Done. Extended simulation saved to three_body_simulation_extended.mp4


Save the trajectory for later

In [ ]:
import pandas as pd
import numpy as np

# Prepare trajectory data for CSV
traj_reshaped = traj.reshape(traj.shape[0], -1) # Reshape to (n_frames, 9)
traj_cols = []
for i in range(masses.shape[0]):
    for coord in ['x', 'y', 'z']:
        traj_cols.append(f'body{i}_{coord}')
df_traj = pd.DataFrame(traj_reshaped, columns=traj_cols)

# Create DataFrames for time and energy
df_times = pd.DataFrame({'time': times})
df_energy = pd.DataFrame({'total_energy': energy})

# Concatenate time-series data into a single DataFrame and save to CSV
df_main_simulation_data = pd.concat([df_times, df_energy, df_traj], axis=1)
df_main_simulation_data.to_csv('simulation_data.csv', index=False)

# Prepare metadata (masses, ejected_body, ejected_at)
metadata_dict = {
    f'mass_{i}': masses[i] for i in range(len(masses))
}
metadata_dict['ejected_body'] = ejected_body
metadata_dict['ejected_at'] = ejected_at
df_metadata = pd.DataFrame([metadata_dict]) # Create a DataFrame from the dictionary
df_metadata.to_csv('simulation_metadata.csv', index=False)

print("Simulation data successfully saved to 'simulation_data.csv' and 'simulation_metadata.csv'.")

In [ ]:
import pandas as pd
import numpy as np
from tqdm.notebook import tqdm # Import tqdm

# Load the simulation data
df_main_simulation_data = pd.read_csv('simulation_data.csv')
df_metadata = pd.read_csv('simulation_metadata.csv')

# Reconstruct times and energy
times_loaded = df_main_simulation_data['time'].values
energy_loaded = df_main_simulation_data['total_energy'].values

# Reconstruct trajectory
traj_cols = [col for col in df_main_simulation_data.columns if col.startswith('body')]
traj_reshaped_loaded = df_main_simulation_data[traj_cols].values
traj_loaded = traj_reshaped_loaded.reshape(traj_reshaped_loaded.shape[0], 3, 3)

# Reconstruct metadata
masses_dict = {k: v for k, v in df_metadata.iloc[0].items() if k.startswith('mass_')}
masses_loaded = np.array([masses_dict[f'mass_{i}'] for i in range(len(masses_dict))])
ejected_body_loaded = int(df_metadata.iloc[0]['ejected_body']) # Convert to int
ejected_at_loaded = df_metadata.iloc[0]['ejected_at']

print("Simulation data loaded from CSV files.")

# Define rendering range (adjust these values as needed)
start_frame_idx = 0  # Start from the beginning
# For example, to render only the first 30% of frames:
end_frame_idx = int(len(traj_loaded) * 0.6) # Adjust this to render a smaller portion of the simulation
# end_frame_idx = None # Render until the end

# Apply slicing to the loaded data
traj_slice = traj_loaded[start_frame_idx:end_frame_idx]
energy_slice = energy_loaded[start_frame_idx:end_frame_idx]
times_slice = times_loaded[start_frame_idx:end_frame_idx]

# Define output video filename for the rendered slice
output_video_filename_sliced = "three_body_simulation_sliced.mp4"

print(f"Rendering video for frames from index {start_frame_idx} to {end_frame_idx if end_frame_idx is not None else 'end'}.")

# Define a tqdm progress bar
with tqdm(total=len(traj_slice), desc="Rendering video frames") as pbar:
    def progress_callback(frame_i, total_frames):
        pbar.update(1)

    # Call the render_video function with the sliced data and the progress callback
    render_video(
        traj_slice, energy_slice, times_slice,
        masses_loaded, ejected_body_loaded,
        output_path=output_video_filename_sliced,
        trail_len=350, fps=120, n_frames_out=18000, # Balanced n_frames_out for faster rendering and smoother trajectories.
        progress_callback=progress_callback
    )

print(f"Sliced video saved to {output_video_filename_sliced}")